# Bloomberg Data Pull — xbbg Examples

**Prerequisites**
- Bloomberg Terminal must be **open and running** (connects via localhost:8194)
- `pip install xbbg`
- `pip install blpapi --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/`

> **Jupyter note:** This notebook uses `await blp.a*()` (async variants) which work natively in Jupyter's async event loop. Do **not** use `asyncio.run()` — Jupyter already runs an event loop.

---

This notebook covers four Bloomberg data functions:
1. **BDP** — current / reference snapshot data
2. **BDH** — historical daily time-series
3. **BDIB** — intraday bars

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xbbg import blp

%matplotlib inline

## 1. BDP — Bloomberg Data Point

Returns a **current snapshot** value for each (ticker, field) pair.  
Useful for: last price, currency, market cap, ratings, fundamental fields.

`xbbg` returns a Narwhals DataFrame. `.to_pandas()` converts it; `.pivot()` reshapes to a clean wide table (tickers as rows, fields as columns).

In [ ]:
bdp_result = (
    (await blp.abdp(
        tickers=["AAPL US Equity", "MSFT US Equity"],
        flds=["PX_LAST", "CURRENCY", "CUR_MKT_CAP"],
    ))
    .to_pandas()
    .pivot(index="ticker", columns="field", values="value")
    .rename_axis(columns=None)
)

bdp_result

## 2. BDH — Bloomberg Data History

Returns a **daily time-series** for one or more tickers and fields between two dates.  
Useful for: backtesting, charting, return calculation.

In [ ]:
bdh_result = (
    (await blp.abdh(
        tickers="SPX Index",
        flds=["PX_LAST", "VOLUME"],
        start_date="2026-03-01",
        end_date="2026-04-06",
    ))
    .to_pandas()
    .pivot(index="date", columns="field", values="value")
    .rename_axis(columns=None)
)

# Convert index to datetime and values to numeric for charting
bdh_result.index = pd.to_datetime(bdh_result.index)
bdh_result = bdh_result.apply(pd.to_numeric, errors="coerce")

bdh_result

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bdh_result["PX_LAST"].plot(ax=ax, title="SPX Index — Daily Close (Mar–Apr 2026)", ylabel="Price", grid=True)
plt.tight_layout()
plt.show()

## 3. BDIB — Intraday Bars

Returns OHLCV bars at a chosen interval (1, 5, 15, 30, 60 minutes, etc.).

**Note:** Bloomberg Terminal typically stores only recent intraday data (~140 days). Adjust `start_datetime` / `end_datetime` to a recent trading session.

In [ ]:
bdib_result = (
    (await blp.abdib(
        ticker="AAPL US Equity",
        event_type="TRADE",
        interval=5,                            # minutes per bar
        start_datetime="2026-04-04 09:30:00",
        end_datetime="2026-04-04 16:00:00",
    ))
    .to_pandas()
)

bdib_result

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
bdib_result["close"].astype(float).plot(ax=ax, title="AAPL — 5-min Intraday Bars (2026-04-04)", ylabel="Price", grid=True)
plt.tight_layout()
plt.show()